# Feature preprocessing pipeline
## Main goal

This notebook builds the schema driven processing pipeline that turns the features created in the extraction pipeline into a clean, fully numeric matrix, ready for model training.

The schema used was defined as the variables were constructed (see the Features Extraction Notebook), or by the variable distribution in the application data (s. The schema consists of the following info:
* type
* expected range
* imputation value
    
## Preproccess data pipeline
    This process assembles a 'scikit-learn' / 'feature-engine' 'Pipeline' that:
1. **Selects** the exact set of features defined in the schema.
2. **Converts data types**, casting numerical features to `float64` and cleaning up
   categorical values into a consistent string format.
3. **Imputes missing values** — categorical variables are filled with an explicit
   `"ND"` placeholder, while numerical variables are filled per the imputation value
   defined for each feature in the schema.
4. **Caps outliers** using the min/max bounds declared in the schema.
5. **Encodes categorical variables**, grouping rare categories together and applying
   target-guided ordinal encoding.
6. **Scales numerical features** (`StandardScaler` by default).

The resulting pipeline is fit once on the training data, serialized with `joblib` for
reuse at inference time, and applied to both the train and test sets to produce the
final preprocessed CSV files used downstream for model training.

> **Note:** the preprocessor is entirely schema-driven — adding, removing, or
> reclassifying a feature only requires updating `data_schema.json`, with no changes
> needed to the pipeline code itself.

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import json
import joblib

#
from pathlib import Path

#Pipeline elements
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from feature_engine.imputation import CategoricalImputer
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.outliers import ArbitraryOutlierCapper
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from functools import partial


pd.set_option('display.max_columns', None)


In [2]:
#Define paths
current_dir = Path.cwd()
ROOT_DIR = current_dir.parent
SCHEMA_DIR = ROOT_DIR / "schema"
MODEL_DIR = ROOT_DIR / "models"
METRICS_DIR = ROOT_DIR / "metrics"

DATA_DIR = ROOT_DIR / "data"
os.chdir(ROOT_DIR)

In [3]:
class ReadWritePaths:
    """
    A class to manage file paths for saving data and related metrics.
    """
    def __init__(self, data_dict):
        self.__dict__.update(data_dict)


In [28]:
class WritePathsPreprocess(ReadWritePaths):
     """
     A class to manage file paths for saving preprocessed data and related metrics.
     """
     def __init__(self, name: str):
        #Make sure paths exist
        DIR = MODEL_DIR / f"{name}"
        Path(DIR).mkdir(parents=True, exist_ok=True)
        for item in ["data", "metrics", "artifacts", "schema"]:
            Path(DIR / item).mkdir(parents=True, exist_ok=True)
        paths = {
            "model_path" : DIR,
             "train_dataset_path" : DIR / 'data' /f'train_{name}_preprocessed.csv',
             "test_dataset_path" : DIR /'data' /f'test_{name}_preprocessed.csv',
             "completeness_report_path" : DIR /'metrics' / f'features_completeness_report{name}.csv',
             "pipeline_path" : DIR /'artifacts' / f'preprocessor_{name}.joblib.dat',
             "ordinal_encoder_dict_path" : DIR / 'schema' / f'ordinal_encoder_dict{name}.json',
         }
        self.__dict__.update(paths)
 # pylint: disable=R0903
class ReadPathsPreprocess(ReadWritePaths):
     """
     A class to manage file paths for reading preprocessor
     """
     def __init__(self, r_suffix: str):
        DIR = MODEL_DIR / f"{name}"
        Path(DIR).mkdir(parents=True, exist_ok=True)
        for item in [ "artifacts", ]:
            Path(DIR / item).mkdir(parents=True, exist_ok=True)
        paths = {
             "pipeline_path" : DIR /'artifacts' / f'preprocessor{r_suffix}.joblib.dat'
         }
        self.__dict__.update(paths)
             


In [7]:
 def load_data_schema(data_schema_path):
     """
     Load a data schema from a JSON file and return it as an ordered dictionary.

     Args:
         data_schema_path (str): The file path to the JSON data schema.

     Returns:
         OrderedDict: The data schema loaded from the JSON file, preserving the order of keys.
     """
     with open(data_schema_path, 'r', encoding='utf-8') as file:
         data_dict = json.load(file, object_pairs_hook=OrderedDict)
     return data_dict

In [8]:
#value
def convert_to_numeric(value):
    """
    Convert a value to a numeric type. If conversion fails, return NaN.

    Parameters:
    -----------
    value : any
        The value to be converted.

    Returns:
    --------
    float
        The numeric representation of the value, or NaN if conversion fails.
    """
    try:
        return pd.to_numeric(value)
    except (ValueError, TypeError):
        return np.nan
#Dataframe
def convert_numerical_features(df, columns):
    """
    Converts numeric features of dataframe to float 64
    """
    df[columns] = df[columns].apply(convert_to_numeric)
    return df
def clean_up(s):
    s = s.astype(str)
    s = s.str.upper()
    s = s.str.strip()
    s = s.str.replace(r'\s+', '_', regex=True)
    s = s.replace(r'\.0$', '', regex=True)
    s = s.replace(r'^NAN$|^$', np.nan, regex=True)
    return s
def convert_categorical_features(df, columns):
    """
    Converts numeric features of dataframe to float 64
    """
    df[columns] = df[columns].astype(
                                                object, 
                                                errors='ignore'
                                                )
    df[columns] = df[columns].fillna('')
    df[columns] = df[columns].apply(clean_up)
                                                
    return df
class Converter:
    """
    Original class for transformers and converters.
    """
    def __init__(self, variables: list[str]):
        self

    def transform(self, X) -> pd.DataFrame:
        """
        Transform method for compatibility with scikit-learn pipelines.

        Parameters:
        -----------
        X : pd.DataFrame
            The input DataFrame.

        Returns:
        --------
        pd.DataFrame
            This instance does nothing for the dataframe.
        """
        return X

    def fit(self, X: pd.DataFrame, y=None) -> 'ManualSelector':
        """
        Fit method for compatibility with scikit-learn pipelines.
        """
        return self
class ManualSelector(Converter):
    """
    A transformer for manually selecting specified variables from a DataFrame.

    Parameters:
    -----------
    variables : List[str]
        The list of variables to select.

    Methods:
    --------
    transform(X: pd.DataFrame) -> pd.DataFrame
        Selects the specified variables from the DataFrame.

    fit(X: pd.DataFrame, y=None) -> ManualSelector
        Fits the transformer, returns self.
    """

    def __init__(self, variables: list[str]):
        self.variables = variables

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Select specified variables from the input DataFrame.

        Parameters:
        -----------
        X : pd.DataFrame
            The input DataFrame.

        Returns:
        --------
        pd.DataFrame
            DataFrame containing only the selected variables.
        """
        return X[self.variables]


class TypeConverter(Converter):
    """
    A transformer to convert data types of numerical, boolean, and categorical features 
    in a DataFrame, including imputations for missing values.

    Parameters:
    -----------
    numerical_features : list[str]
        list of numerical feature names.
    categorical_features : list[str]
        List of categorical feature names.

    Methods:
    --------
    transform(X: pd.DataFrame) -> pd.DataFrame
        Transforms the data types of the input DataFrame.
    fit(X: pd.DataFrame, y=None) -> TypeConverter
        Fits the transformer, returns self.
    """
    def __init__(
        self,
        numerical_features: list[str],
        categorical_features: list[str],
    ):
        self.numerical_features = numerical_features
        self.categorical_features = categorical_features
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Transforms the data types of numerical, boolean, and categorical features.

        Parameters:
        -----------
        X : pd.DataFrame
            The input DataFrame to be transformed.

        Returns:
        --------
        pd.DataFrame
            Transformed DataFrame with updated data types.
        """
        # Converting numerical features to float64
        X = convert_numerical_features(X, self.numerical_features)
        
    
        # Converting categorical variables to object and cleaning categories
        X = convert_categorical_features(X, self.categorical_features)
        

        return X


In [9]:
def get_feature_lists(data_dict):
    """
     Extract feature lists and imputation rules from a data schema dictionary.

     This function categorizes features into numerical, boolean, and categorical types 
     and identifies additional imputation and capping rules based on the provided schema.

     Args:
         data_dict (dict): A dictionary where keys are feature names and values are lists 
                           containing attributes of the features. The attributes include 
                           information such as type (e.g., 'numerical', 'boolean'), 
                           imputation strategy, and capping values.

    Returns:
         tuple: A tuple containing the following elements:
             - numerical_features (list): List of numerical feature names.
             - categorical_features (list): List of categorical feature names.
             - feature_names (list): List of all feature names.
     """
    features = {}
    features["numerical"] = [
         feature for feature, attributes in data_dict.items()
         if attributes['type'] == 'numeric'
    ]
    features["categorical"] = [
         feature for feature, attributes in data_dict.items()
         if attributes['type'] == 'categorical'
    ]
    features["impute"] = {}
    impute_values = list(set(attributes["impute_value"] for attributes in data_dict.values()))

    for value in impute_values:
        features["impute"][value] = [
             feature for feature, attributes in data_dict.items()
             if attributes['impute_value'] == value
        ]
    features["max"] = {feature:json.loads(attributes['range'])[1]
                       for feature, attributes in data_dict.items()
                 if attributes['type'] == 'numeric' and json.loads(attributes['range'])[1] is not None

}
    features["min"] = {feature:json.loads(attributes['range'])[0] 
                       for feature, attributes in data_dict.items()
                 if attributes['type'] == 'numeric' and json.loads(attributes['range'])[0] is not None

}
    feature_names = list(data_dict.keys())
    
    return (
         features,
         feature_names
     )


In [23]:


def recast_numeric(df, columns):
    df = df.copy()
    df[columns] = df[columns].astype(float)
    return df
def create_initial_pipeline(data_dict,
                            max_cap = True,
                            min_cap = True):
    """
     Create an initial preprocessing pipeline based on the data schema.

     Args:
         data_dict (dict): A dictionary specifying feature types and imputation rules, 
                           where keys are feature names and values are lists of attributes.
         scaler (str): Name of the scaler to be applied to numerical features.

     Returns:
         tuple: A tuple containing:
             - Pipeline: A scikit-learn pipeline object with the defined preprocessing steps.
             - numerical_features (list): List of numerical feature names for further processing.
    """
    (
         features,
         feature_names
    ) = get_feature_lists(data_dict)
    steps = [
         ('manual_selector', ManualSelector(variables=feature_names)),
         ('type_converter', TypeConverter(
             numerical_features = features["numerical"],
             categorical_features = features["categorical"],

         )),
         ('categorical_imputer', CategoricalImputer(
             variables=features["categorical"],
             imputation_method='missing',
             fill_value='ND',
             ignore_format=True,
             return_object=True
         ))]
    #Impute values
    for value in features["impute"]:
        steps = steps + [
            (f'simple_imputer_{value}', SklearnTransformerWrapper(
             SimpleImputer(
                            missing_values=np.nan,
                            strategy='constant',
                            fill_value= value,
                            keep_empty_features=True),
             variables=features["impute"][value]
         ))
        ] 
    steps = steps + [
     ('numeric_recast', FunctionTransformer(
    partial(recast_numeric, columns=features["numerical"])
        ))]
    #Are the caps?
    if max_cap == True:
        max_capping_dict = features["max"]
    if min_cap == True:
        min_capping_dict = features["min"]
    if max_cap == True or min_cap == True:
        steps = steps + [('outlier_capper',
                        ArbitraryOutlierCapper(
                        max_capping_dict=max_capping_dict,
                        min_capping_dict=min_capping_dict,
                        ))]

    steps = steps + [('rare_label_encoder', RareLabelEncoder(
             variables= features["categorical"], tol=0.012, n_categories=5,
             max_n_categories=20, replace_with='OTHER', ignore_format=True
         )),
         ('ordinal_encoder', OrdinalEncoder(
             variables=features["categorical"], encoding_method='ordered',
             ignore_format=True
         ))
                    ]       
    steps = steps + [('simple_imputer', SklearnTransformerWrapper(
             SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1),
             variables=feature_names    
    ))]

    return Pipeline(steps, verbose=True), features["numerical"]

In [26]:
def get_preprocessor(
     x: pd.DataFrame,
     y: pd.Series,
     data_schema_path: str,
     scaler='StandardScaler',
     write_paths: object = None
 ):
     """
     Create and save a preprocessing pipeline for feature engineering and selection.

     This function constructs a preprocessing pipeline that includes data type conversion, 
     feature imputation, scaling, and optional feature selection based on importance scores. 
     It processes the input features (X) using the specified steps and saves the pipeline 
     and encoding dictionaries to the provided paths.

     Args:
         x (pd.DataFrame): Input feature matrix.
         y (pd.Series): Target variable.
         data_schema_path (str): Path to the data schema JSON file.

         n_estimators (int): Number of estimators for ensemble methods. Defaults to 100.
         max_features (int): Maximum number of features to select.
         threshold (int): Threshold for feature importance scores. Defaults to 0.
         scaler (str): Scaling method to apply to numerical features .
         random_state (int): Random seed for reproducibility. Defaults to 1112.
         write_paths (object): An object containing paths for saving the pipeline.

     Returns:
         final_pipeline (Pipeline): A scikit-learn pipeline object with preprocessing 
             and feature selection steps.
         column_names (list): List of feature names selected or retained after preprocessing.
     """
     data_dict = load_data_schema(data_schema_path)
     scaler_ = {
         "StandardScaler": StandardScaler(),
         "MinMaxScaler": MinMaxScaler(),
         "None": None
     }[scaler]

     initial_pipeline, numerical_features = create_initial_pipeline(
         data_dict
     )
     feature_names = list(data_dict.keys())   
     
     initial_pipeline.fit(x, y)
     ordinal_encoder_dict = getattr(initial_pipeline['ordinal_encoder'], 'encoder_dict_', {})

     x_preprocessed = initial_pipeline.transform(x)
     print('x_preprocessed inital pipeline: Done')

     if scaler_:
         scaler_pipeline = Pipeline(
             [('scaler', SklearnTransformerWrapper(scaler_, variables=numerical_features))],
             verbose=True
         )
         scaler_pipeline.fit(x_preprocessed)
         x_preprocessed = scaler_pipeline.transform(x_preprocessed)

         initial_pipeline = Pipeline([
             ('initial_pipeline', initial_pipeline),
             ('scaler', scaler_pipeline)
         ], verbose=True)

         
     final_pipeline = initial_pipeline
     column_names = feature_names
     loc_dir = data_schema_path.parent
     schema_file = data_schema_path.name

     new_schema_file = f'{schema_file.split(".")[0]}_train.{schema_file.split(".")[1]}'
     assert target_col not in column_names 
     for item in other_metrics:
         assert item not in column_names
     if isinstance(write_paths, ReadWritePaths):
        with open( loc_dir / new_schema_file,
                    'w',
                    encoding='utf-8') as file_schema:
            json.dump(data_dict, file_schema, separators=(',', ':'), indent=2)
        joblib.dump(final_pipeline,
                         write_paths.pipeline_path)
     
        with open(write_paths.ordinal_encoder_dict_path, 'w', encoding='utf-8') as file_encoder:
             json.dump(ordinal_encoder_dict, file_encoder, indent=4)

        print(
         f'Preprocessor saved at {write_paths.pipeline_path} '
         f'& Ordinal encoder dict saved at {write_paths.ordinal_encoder_dict_path}!'
         )
     else:
         print("No write_paths provided")
     return final_pipeline, column_names

In [12]:
 def save_preprocessed_data(
         x, y, other_metrics, preprocessor, column_names, target, path, save=True):
     """
     Save the preprocessed dataset.

     This function applies the provided preprocessor to the input features 
     (X) and combines the transformed data with the target variable (y) 
     and any specified other metrics. The resulting DataFrame is then 
     saved as a CSV file at the specified path if `save` is set to True. 
     If `save` is False, the function returns the preprocessed DataFrame 
     instead of saving it.

     Args:
         x (pd.DataFrame): The input features to be preprocessed.
         y (pd.Series): The target variable associated with the input features.
         other_metrics (list): A list of other metrics to include in the final DataFrame.
         preprocessor: The preprocessor object used to transform the input features.
         column_names (list): The names of the columns in the transformed DataFrame.
         target (str): The name of the target variable to include in the final DataFrame.
         path (str): The file path where the preprocessed dataset will be saved.
         save (bool): Flag indicating whether to save the DataFrame as a CSV file.

     Returns:
         pd.DataFrame or None: Returns the preprocessed DataFrame if `save` is False; 
                               otherwise, returns None.
     """
     

     data_preprocessed = pd.DataFrame(
         preprocessor.transform(x), columns=column_names, index=x.index)
     data_preprocessed[target] = y
     if other_metrics is None:
         other_metrics = []
     data_preprocessed[other_metrics] = x[other_metrics]
     data_preprocessed = data_preprocessed[other_metrics + [target] + column_names]
     if save:
         data_preprocessed.to_csv(path, index = False)
         print(f'Dataset saved at {path}!')
    
     return data_preprocessed


 def get_column_names(preprocessor):
     """
     Retrieve column names from a preprocessing object.

     Parameters:
     ----------
     preprocessor : object
         The preprocessor object, which is expected to contain methods or attributes 
         that provide the names of the processed features.

     Returns:
     -------
     list
         A list of column names derived from the preprocessor.
     """
     try:
         return preprocessor[-1].get_feature_names_out()
     except AttributeError:
         pass

     for attr in ['out_var', 'variables', 'poly_features', 'best_features']:
         try:
             return getattr(preprocessor[0], attr, preprocessor[0].variables)
         except AttributeError:
             continue

     raise RuntimeError("Unable to determine column names from preprocessor.")

In [13]:
#Load Data And Schema
data_train_path = DATA_DIR / "application_train_all_features.csv"
data_test_path = DATA_DIR / "application_test_all_features.csv"
data_schema_path = SCHEMA_DIR/"data_schema.json"
model_name = "model_nb" #lighter for faster execution


In [14]:
#Load Data and Schema

target_col = "TARGET"
other_metrics = ["SK_ID_CURR"] #ID columns and columns not related with model

In [29]:
path_preprocessor = WritePathsPreprocess(model_name)


In [27]:
schema = load_data_schema(data_schema_path)
train_final = pd.read_csv(data_train_path).sample(5000)
test_final = pd.read_csv(data_test_path).sample(5000)
X = train_final
y = train_final[target_col]
#Start preprocessor
path_preprocessor = WritePathsPreprocess(model_name)
preprocessor, column_names = get_preprocessor(
     x = X,
     y = y,
     data_schema_path = data_schema_path,
   
     write_paths = path_preprocessor
 )
#Preprocesed data
save_preprocessed_data(
        x = X,
        y = y,
        preprocessor = preprocessor,
        column_names = column_names,
        other_metrics = other_metrics,
        target = target_col,
        path = path_preprocessor.train_dataset_path,
        save = True
)
save_preprocessed_data(
        x = test_final,
        y = test_final["TARGET"],
        preprocessor = preprocessor,
        column_names = column_names,
        other_metrics = other_metrics,
        target = target_col,
        path = path_preprocessor.test_dataset_path,
        save = True
)
model_columns = get_column_names(preprocessor)
#preprocessor[-1].get_feature_names_out()

[Pipeline] .. (step 1 of 10) Processing manual_selector, total=   0.0s
[Pipeline] ... (step 2 of 10) Processing type_converter, total=   0.3s
[Pipeline]  (step 3 of 10) Processing categorical_imputer, total=   0.0s
[Pipeline] . (step 4 of 10) Processing simple_imputer_0, total=   0.2s
[Pipeline]  (step 5 of 10) Processing simple_imputer_ND, total=   0.1s
[Pipeline] ... (step 6 of 10) Processing numeric_recast, total=   0.1s
[Pipeline] ... (step 7 of 10) Processing outlier_capper, total=   0.1s


/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_contract_type is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_status is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_payment_type is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documen

[Pipeline]  (step 8 of 10) Processing rare_label_encoder, total=   0.1s
[Pipeline] .. (step 9 of 10) Processing ordinal_encoder, total=   0.0s
[Pipeline] .. (step 10 of 10) Processing simple_imputer, total=   0.0s
x_preprocessed inital pipeline: Done
[Pipeline] ............ (step 1 of 1) Processing scaler, total=   0.0s
Preprocessor saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/artifacts/preprocessor_model_nb.joblib.dat & Ordinal encoder dict saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/schema/ordinal_encoder_dictmodel_nb.json!
Dataset saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/data/train_model_nb_preprocessed.csv!


/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) freq_latest_status_cash, freq_currency, NAME_INCOME_TYPE, FLAG_DOCUMENT_12.
  warnings.warn(


Dataset saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/data/test_model_nb_preprocessed.csv!
